In [1]:
import pandas as pd
import numpy as np
import warnings, gc, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from IPython.display import display
pd.set_option('display.max_columns', None)
!rm -rf /kaggle/working/

rm: cannot remove '/kaggle/working/': Device or resource busy


In [2]:
train = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-database/train.csv")
test  = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-database/test.csv")
train.drop(columns=["id"],axis=1,inplace=True)

print("Check Out Train DaTA Null Values: ",train.isnull().sum())
print(f"Train Data Shape: {train.shape}")
print(f"Train Data INFO: {train.info()}")

def bank_feature_engineering(df):
    df = df.copy()
  
    df['no_previous_contact'] = (df['pdays'].isna()).astype(int)
    df['ever_contacted']      = 1 - df['no_previous_contact']
    df['duration_hour'] = df['duration'] // 60
    df['duration_min']  = df['duration'] % 60
    df['is_long_call']  = (df['duration'] > 500).astype(int)
    df['is_very_long_call'] = (df['duration'] > 900).astype(int)
    df['is_young'] = (df['age'] <= 30).astype(int)
    df['is_senior'] = (df['age'] >= 60).astype(int)
    df['age_x_balance'] = df['age'] * df['balance'].clip(lower=0)
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['high_balance'] = (df['balance'] > 3000).astype(int)
    df['negative_balance'] = (df['balance'] < 0).astype(int)
    df['was_contacted_before'] = (df['previous'] > 0).astype(int)
    df['many_campaigns'] = (df['campaign'] > 4).astype(int)
    df['previous_per_campaign'] = df['previous'] / (df['campaign'] + 1)
    month_order = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6,
                   'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}
    df['month_num'] = df['month'].map(month_order)
    df['is_quarter_end'] = df['month'].isin(['mar', 'jun', 'sep', 'dec']).astype(int)
    df['job_education'] = df['job'] + "_" + df['education']
    df['job_marital']   = df['job'] + "_" + df['marital']
    df['balance_bin']   = pd.qcut(df['balance'], q=10, duplicates='drop').astype(str)
    df['poutcome_success_before'] = (df['poutcome'] == 'success').astype(int)
    df['poutcome_failure_before'] = (df['poutcome'] == 'failure').astype(int)
    df['contact_month'] = df['contact'] + "_" + df['month']
    df['debt_burden'] = df['housing'].apply(lambda x: 1 if x=='yes' else 0) + df['loan'].apply(lambda x: 1 if x=='yes' else 0)
    df['call_in_best_month'] = df['month'].isin(['mar', 'sep', 'oct', 'dec']).astype(int)
    df['student_or_retired'] = df['job'].isin(['student', 'retired']).astype(int)
    return df


train = bank_feature_engineering(train)
test=bank_feature_engineering(test)

balance_bins = [-10000, 0, 500, 1000, 3000, 10000, 100000]  # adjust based on your dataset
balance_labels = ['neg', 'very_low', 'low', 'medium', 'high', 'very_high']
train['balance_bin'] = pd.cut(train['balance'], bins=balance_bins, labels=balance_labels, include_lowest=True)
train['balance_bin'] = train['balance_bin'].astype(str)

test['balance_bin'] = pd.cut(test['balance'], bins=balance_bins, labels=balance_labels, include_lowest=True)
# test['balance_bin'] = test['balance_bin'].astype(str)


categorical_cols = train.select_dtypes(include='object').columns.tolist()

if 'y' in categorical_cols:
    categorical_cols.remove('y')  


label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
  
    test[col] = le.transform(test[col])  
    
    label_encoders[col] = le


numeric_cols = ['age','balance','duration','campaign','previous','age_x_balance']
scaler = StandardScaler()
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])
test[numeric_cols] = scaler.transform(test[numeric_cols])
Id=test.id
test.drop(columns=["id"],axis=1,inplace=True)

print("Display Train Data:")
display(train.head())
print("#"*130)
print("\n")
print("Display Test Data:")
display(test.head())

Check Out Train DaTA Null Values:  age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64
Train Data Shape: (750000, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 17 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   age        750000 non-null  int64 
 1   job        750000 non-null  object
 2   marital    750000 non-null  object
 3   education  750000 non-null  object
 4   default    750000 non-null  object
 5   balance    750000 non-null  int64 
 6   housing    750000 non-null  object
 7   loan       750000 non-null  object
 8   contact    750000 non-null  object
 9   day        750000 non-null  int64 
 10  month      750000 non-null  object
 11  duration   750000 non-null  in

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,no_previous_contact,ever_contacted,duration_hour,duration_min,is_long_call,is_very_long_call,is_young,is_senior,age_x_balance,has_balance,high_balance,negative_balance,was_contacted_before,many_campaigns,previous_per_campaign,month_num,is_quarter_end,job_education,job_marital,balance_bin,poutcome_success_before,poutcome_failure_before,contact_month,debt_burden,call_in_best_month,student_or_retired
0,0.106310,9,1,1,0,-0.422083,0,0,0,25,1,-0.510829,0.155597,-1,-0.223475,3,0,0,1,1,57,0,0,0,0,-0.394414,1,0,0,0,0,0.0,8,0,37,28,5,0,0,1,0,0,0
1,-0.289776,1,1,1,0,-0.243316,0,0,2,18,6,-0.261338,-0.580100,-1,-0.223475,3,0,0,1,3,5,0,0,0,0,-0.250238,1,0,0,0,0,0.0,6,1,5,4,1,0,0,30,0,0,0
2,-0.487819,1,1,1,0,-0.212287,1,0,2,14,8,-0.532843,-0.212251,-1,-0.223475,3,0,0,1,1,51,0,0,0,0,-0.234201,1,0,0,0,0,0.0,5,0,5,4,1,0,0,32,1,0,0
3,-1.379012,8,2,1,0,-0.412563,1,0,2,28,8,-0.903409,-0.212251,-1,-0.223475,3,0,0,1,0,10,0,0,1,0,-0.389737,1,0,0,0,0,0.0,5,0,33,26,5,0,0,32,1,0,1
4,-1.478033,9,1,1,0,-0.111092,1,0,0,3,3,2.369319,-0.580100,-1,-0.223475,3,1,0,1,15,2,1,1,1,0,-0.223394,1,0,0,0,0,0.0,2,0,37,28,1,0,0,3,1,0,0


##################################################################################################################################


Display Test Data:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,no_previous_contact,ever_contacted,duration_hour,duration_min,is_long_call,is_very_long_call,is_young,is_senior,age_x_balance,has_balance,high_balance,negative_balance,was_contacted_before,many_campaigns,previous_per_campaign,month_num,is_quarter_end,job_education,job_marital,balance_bin,poutcome_success_before,poutcome_failure_before,contact_month,debt_burden,call_in_best_month,student_or_retired
0,-0.883905,1,1,1,0,0.068028,1,0,2,21,8,-0.118248,-0.580100,-1,-0.223475,3,0,1,3,44,0,0,0,0,-0.061592,1,0,0,0,0,0.0,5,0,5,4,2,0,0,32,1,0,0
1,0.304353,4,1,2,0,-0.416441,1,0,0,3,0,1.209922,-0.212251,-1,-0.223475,3,0,1,9,46,1,0,0,0,-0.389033,1,0,0,0,0,0.0,4,0,18,13,5,0,0,0,1,0,0
2,-0.487819,6,1,0,0,-0.408332,1,1,0,13,8,-0.532843,-0.212251,-1,-0.223475,3,0,1,1,51,0,0,0,0,-0.384206,1,0,0,0,0,0.0,5,0,24,19,5,0,0,8,2,0,0
3,1.690653,1,1,1,0,-0.911136,1,1,2,29,8,-0.481477,-0.580100,-1,-0.223475,3,0,1,2,5,0,0,0,0,-0.396617,0,0,1,0,0,0.0,5,0,5,4,3,0,0,32,2,0,0
4,-1.279990,9,2,1,0,0.263014,1,0,0,22,5,-0.276014,-0.580100,-1,-0.223475,3,0,1,3,1,0,0,1,0,0.012571,1,0,0,0,0,0.0,7,0,37,29,2,0,0,5,1,0,0


In [3]:
features = [c for c in train.columns if c not in ['id', 'y']]

params_lgb = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 128,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
    'max_depth': -1,
    'min_data_in_leaf': 100,
    'lambda_l1': 0.1,
    'lambda_l2': 0.5,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train))
test_preds_lgb = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(train, train['y'])):
    print(f"\nFold {fold+1}/10")
    X_trn, y_trn = train.iloc[trn_idx][features], train.iloc[trn_idx]['y']
    X_val, y_val = train.iloc[val_idx][features], train.iloc[val_idx]['y']
    
    dtrain = lgb.Dataset(X_trn, y_trn)
    dvalid = lgb.Dataset(X_val, y_val, reference=dtrain)
    
    model = lgb.train(params_lgb,dtrain,num_boost_round=5000,valid_sets=[dtrain, dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds_lgb += model.predict(test[features]) / skf.n_splits
    
    print(f"Fold {fold+1} AUC: {roc_auc_score(y_val, oof_preds[val_idx]):.6f}")

print(f"\nOverall OOF AUC: {roc_auc_score(train['y'], oof_preds):.6f}")



Fold 1/10
Training until validation scores don't improve for 100 rounds
[200]	training's auc: 0.973626	valid_1's auc: 0.971227
[400]	training's auc: 0.978261	valid_1's auc: 0.972348
[600]	training's auc: 0.981529	valid_1's auc: 0.972795
[800]	training's auc: 0.984135	valid_1's auc: 0.972949
Early stopping, best iteration is:
[871]	training's auc: 0.984977	valid_1's auc: 0.973021
Fold 1 AUC: 0.973021

Fold 2/10
Training until validation scores don't improve for 100 rounds
[200]	training's auc: 0.973755	valid_1's auc: 0.969854
[400]	training's auc: 0.978367	valid_1's auc: 0.970985
[600]	training's auc: 0.981572	valid_1's auc: 0.971376
[800]	training's auc: 0.984193	valid_1's auc: 0.97158
[1000]	training's auc: 0.986438	valid_1's auc: 0.971612
[1200]	training's auc: 0.988317	valid_1's auc: 0.971681
Early stopping, best iteration is:
[1139]	training's auc: 0.987779	valid_1's auc: 0.971687
Fold 2 AUC: 0.971687

Fold 3/10
Training until validation scores don't improve for 100 rounds
[200]	t

In [4]:
params_cb = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'learning_rate': 0.05,
    'iterations': 5000,
    'depth': 8,
    'l2_leaf_reg': 3,
    'random_strength': 0.8,
    'bagging_temperature': 0.7,
    'random_seed': 42,
    'task_type': 'GPU',
    'devices': '0',
    'early_stopping_rounds': 400,
    'verbose': 500
}

test_preds_cb = np.zeros(len(test))
oof_preds_cb = np.zeros(len(train))

for fold, (trn_idx, val_idx) in enumerate(skf.split(train, train['y'])):
    print(f"\nCAT Fold {fold+1}/10")
    
    X_trn = train.iloc[trn_idx][features].values
    X_val = train.iloc[val_idx][features].values
    y_trn = train.iloc[trn_idx]['y'].values
    y_val = train.iloc[val_idx]['y'].values
    
    model = cb.CatBoost(params_cb)
    model.fit(X_trn, y_trn, eval_set=(X_val, y_val), use_best_model=True, verbose=500)
    
    oof_preds_cb[val_idx] = model.predict(X_val, prediction_type='Probability')[:, 1]
    test_preds_cb += model.predict(test[features].values, prediction_type='Probability')[:, 1] / skf.n_splits

print(f"\nCatBoost OOF AUC: {roc_auc_score(train['y'], oof_preds_cb):.6f}")



CAT Fold 1/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9416739	best: 0.9416739 (0)	total: 4.88s	remaining: 6h 46m 59s
500:	test: 0.9672755	best: 0.9672755 (500)	total: 12s	remaining: 1m 47s
1000:	test: 0.9690064	best: 0.9690064 (1000)	total: 19.1s	remaining: 1m 16s
1500:	test: 0.9698547	best: 0.9698547 (1500)	total: 26.2s	remaining: 1m 1s
2000:	test: 0.9703693	best: 0.9703712 (1987)	total: 33.3s	remaining: 49.9s
2500:	test: 0.9706897	best: 0.9706909 (2496)	total: 40.4s	remaining: 40.4s
3000:	test: 0.9709498	best: 0.9709498 (3000)	total: 47.5s	remaining: 31.7s
3500:	test: 0.9710876	best: 0.9710923 (3485)	total: 54.7s	remaining: 23.4s
4000:	test: 0.9711775	best: 0.9711834 (3971)	total: 1m 1s	remaining: 15.4s
4500:	test: 0.9712728	best: 0.9712775 (4493)	total: 1m 8s	remaining: 7.64s
4999:	test: 0.9712695	best: 0.9712955 (4807)	total: 1m 16s	remaining: 0us
bestTest = 0.971295476
bestIteration = 4807
Shrink model to first 4808 iterations.

CAT Fold 2/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9386275	best: 0.9386275 (0)	total: 14.7ms	remaining: 1m 13s
500:	test: 0.9661230	best: 0.9661230 (500)	total: 7.16s	remaining: 1m 4s
1000:	test: 0.9680768	best: 0.9680768 (1000)	total: 14.2s	remaining: 56.9s
1500:	test: 0.9688696	best: 0.9688731 (1497)	total: 21.3s	remaining: 49.7s
2000:	test: 0.9693759	best: 0.9693759 (2000)	total: 28.5s	remaining: 42.7s
2500:	test: 0.9696463	best: 0.9696477 (2499)	total: 35.6s	remaining: 35.6s
3000:	test: 0.9698246	best: 0.9698305 (2985)	total: 42.8s	remaining: 28.5s
3500:	test: 0.9699457	best: 0.9699457 (3499)	total: 49.9s	remaining: 21.4s
4000:	test: 0.9700272	best: 0.9700316 (3993)	total: 57.1s	remaining: 14.3s
4500:	test: 0.9700885	best: 0.9700930 (4491)	total: 1m 4s	remaining: 7.13s
bestTest = 0.9700929523
bestIteration = 4491
Shrink model to first 4492 iterations.

CAT Fold 3/10


...

0:	test: 0.9411466	best: 0.9411466 (0)	total: 15.8ms	remaining: 1m 18s
500:	test: 0.9659564	best: 0.9659564 (500)	total: 7.14s	remaining: 1m 4s
1000:	test: 0.9677029	best: 0.9677029 (1000)	total: 14.3s	remaining: 57s
1500:	test: 0.9685587	best: 0.9685587 (1500)	total: 21.4s	remaining: 49.9s
2000:	test: 0.9690133	best: 0.9690146 (1999)	total: 28.7s	remaining: 43s
2500:	test: 0.9693189	best: 0.9693217 (2498)	total: 35.9s	remaining: 35.9s
3000:	test: 0.9695320	best: 0.9695358 (2985)	total: 43.1s	remaining: 28.7s
3500:	test: 0.9695922	best: 0.9695922 (3500)	total: 50.2s	remaining: 21.5s
4000:	test: 0.9697155	best: 0.9697191 (3917)	total: 57.4s	remaining: 14.3s
4500:	test: 0.9698150	best: 0.9698218 (4489)	total: 1m 4s	remaining: 7.16s
4999:	test: 0.9698945	best: 0.9698992 (4994)	total: 1m 11s	remaining: 0us
bestTest = 0.9698992372
bestIteration = 4994
Shrink model to first 4995 iterations.

CAT Fold 9/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9399146	best: 0.9399146 (0)	total: 16.5ms	remaining: 1m 22s
500:	test: 0.9660761	best: 0.9660761 (500)	total: 7.1s	remaining: 1m 3s
1000:	test: 0.9679468	best: 0.9679468 (1000)	total: 14.2s	remaining: 56.8s
1500:	test: 0.9688193	best: 0.9688194 (1498)	total: 21.3s	remaining: 49.8s
2000:	test: 0.9692951	best: 0.9692951 (2000)	total: 28.5s	remaining: 42.7s
2500:	test: 0.9695632	best: 0.9695632 (2500)	total: 35.7s	remaining: 35.6s
3000:	test: 0.9698185	best: 0.9698194 (2993)	total: 42.7s	remaining: 28.5s
3500:	test: 0.9699415	best: 0.9699444 (3492)	total: 49.8s	remaining: 21.3s
4000:	test: 0.9700314	best: 0.9700328 (3993)	total: 57s	remaining: 14.2s
4500:	test: 0.9700569	best: 0.9700769 (4321)	total: 1m 4s	remaining: 7.11s
bestTest = 0.970076859
bestIteration = 4321
Shrink model to first 4322 iterations.

CAT Fold 10/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9381632	best: 0.9381632 (0)	total: 15.6ms	remaining: 1m 17s
500:	test: 0.9652832	best: 0.9652832 (500)	total: 7.08s	remaining: 1m 3s
1000:	test: 0.9671523	best: 0.9671523 (1000)	total: 14.1s	remaining: 56.5s
1500:	test: 0.9681240	best: 0.9681240 (1500)	total: 21.3s	remaining: 49.8s
2000:	test: 0.9686307	best: 0.9686321 (1996)	total: 28.5s	remaining: 42.7s
2500:	test: 0.9689312	best: 0.9689342 (2487)	total: 35.7s	remaining: 35.7s
3000:	test: 0.9691854	best: 0.9691859 (2993)	total: 42.8s	remaining: 28.5s
3500:	test: 0.9693322	best: 0.9693362 (3474)	total: 50s	remaining: 21.4s
4000:	test: 0.9694548	best: 0.9694628 (3935)	total: 57.1s	remaining: 14.3s
4500:	test: 0.9695277	best: 0.9695341 (4461)	total: 1m 4s	remaining: 7.13s
4999:	test: 0.9696082	best: 0.9696143 (4979)	total: 1m 11s	remaining: 0us
bestTest = 0.969614327
bestIteration = 4979
Shrink model to first 4980 iterations.

CatBoost OOF AUC: 0.969950


In [5]:
best_auc = 0
best_w = None

for w in np.linspace(0, 1, 101):
    blend = w * oof_preds + (1-w) * oof_preds_cb
    auc = roc_auc_score(train['y'], blend)
    if auc > best_auc:
        best_auc = auc
        best_w = w

print("Best weight:", best_w, "AUC:", best_auc)


Best weight: 0.87 AUC: 0.9717150391048631


In [6]:
final_pred = 0.86 * test_preds_lgb + 0.14 * test_preds_cb
submission=pd.DataFrame({"id":Id,"y":final_pred})
submission.to_csv('submission.csv', index=False)
print("Submission saved!")
submission.head(5)

Submission saved!


,id,y
0,750000,0.002543
1,750001,0.072556
2,750002,0.000191
3,750003,0.000024
4,750004,0.010382
